## IsTheWorldReadyForTheNextPandemic - Mini Project
### Project handled by Liza, Ravit, Hagit and Hodaya

**This notebook includes assembling the feature matrix of the model**



---
## Part 3 · Assembling the Featue Matrix


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Core imports.
import numpy as np
np.random.seed(42)  # For reproducibility
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set the path where your DataSet CSVs located.
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'


## Assembling the feature Matrix:
1. X1 - מיטות אשפוז לכל 1,000 תושבים (%)
2. x2 - רופאים לכל 1,000 תושבים (%)
3. X3 - ציון "Early Detection & Reporting" ממדד ה-GHS
4. X4 - שיעור בדיקות קורונה בשנת 2021 ל-1,000 איש (%)
5. X5 - ציון מדד GHSI הכללי
6. X6 - אחוז משתמשי האינטרנט במדינה (%)
7. X7 - אחוז בני +60 באוכלוסייה (%)


In [3]:
import os
import pandas as pd

FOLDER_PATH = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

# 1. בדיקת עמודות קובץ הבריאות
print("--- עמודות בקובץ הבריאות (מיזוג 1) ---")
try:
    df_health = pd.read_csv(os.path.join(FOLDER_PATH, 'cleaned_health_demographics_final.csv'))
    print(df_health.columns.tolist()[:15]) # 15 הראשונות
except Exception as e:
    print(f"שגיאה בטעינת קובץ בריאות: {e}")

# 2. בדיקת עמודות קובץ האפידמיולוגיה (הקובץ שתיקנו עכשיו)
print("\n--- עמודות בקובץ האפידמיולוגיה (מיזוג 4) ---")
try:
    df_epi = pd.read_csv(os.path.join(FOLDER_PATH, 'merged_epidemiology_demographics_yearly.csv'))
    print(df_epi.columns.tolist()[:15])
except Exception as e:
    print(f"שגיאה בטעינת קובץ אפידמיולוגיה: {e}")

# 3. בדיקת עמודות קובץ ה-GHS והטכנולוגיה שבזיכרון
print("\n--- עמודות במשתנה הזיכרון df_ict_ghs_cleaned (מיזוג 3) ---")
try:
    print(df_ict_ghs_cleaned.columns.tolist()[:15])
except Exception as e:
    print(f"משתנה df_ict_ghs_cleaned לא נמצא בזיכרון! שגיאה: {e}")


--- עמודות בקובץ הבריאות (מיזוג 1) ---
['location_key', 'country_code', 'country_name', 'aggregation_level', 'population', 'population_male', 'population_female', 'population_rural', 'population_urban', 'population_largest_city', 'population_clustered', 'population_density', 'human_development_index', 'population_age_00_09', 'population_age_10_19']

--- עמודות בקובץ האפידמיולוגיה (מיזוג 4) ---
['location_key', 'YEAR', 'country_code', 'country_name', 'aggregation_level', 'new_confirmed', 'new_deceased', 'new_recovered', 'new_tested', 'cumulative_confirmed', 'cumulative_deceased', 'cumulative_recovered', 'cumulative_tested', 'population', 'population_male']

--- עמודות במשתנה הזיכרון df_ict_ghs_cleaned (מיזוג 3) ---
משתנה df_ict_ghs_cleaned לא נמצא בזיכרון! שגיאה: name 'df_ict_ghs_cleaned' is not defined


In [11]:
import os
import pandas as pd
import numpy as np

# הגדרת נתיב התיקייה בדרייב
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

# יצירת התיקייה במידה והיא עדיין לא קיימת בדרייב
os.makedirs(data_dir, exist_ok=True)

# =====================================================================
# 1. טעינת קבצי המקור
# =====================================================================
print("1. טוען את קבצי המקור המעודכנים...")
df_master_epi_demo = pd.read_csv(os.path.join(data_dir, 'merged_epidemiology_demographics_yearly.csv'))
df_master_epi_demo.columns = df_master_epi_demo.columns.str.strip()

df_health_profile = pd.read_csv(os.path.join(data_dir, 'cleaned_health_demographics_final.csv'))
df_health_profile.columns = df_health_profile.columns.str.strip()

# טעינת קובץ הטכנולוגיה/GHS ישירות מהדרייב
try:
    df_ict_ghs_cleaned = pd.read_csv(os.path.join(data_dir, 'merged_tech_ghs_final.csv'))
except FileNotFoundError:
    print("קובץ הטכנולוגיה לא נמצא בשם ברירת המחדל, מציג קבצים קיימים ומנסה לאתר...")
    all_files = os.listdir(data_dir)
    tech_file = [f for f in all_files if 'ict' in f.lower() or 'tech' in f.lower() or 'ghs' in f.lower()]
    if tech_file:
        df_ict_ghs_cleaned = pd.read_csv(os.path.join(data_dir, tech_file[0]))
    else:
        raise FileNotFoundError("לא נמצא קובץ מתאים לנתוני ICT/GHS בתיקייה בדרייב.")

df_tech_ghs_raw = df_ict_ghs_cleaned.copy()
df_tech_ghs_raw.columns = df_tech_ghs_raw.columns.str.strip()

# וידוא שכל קבצי המקור כוללים את location_key כעמודה רגילה ולא כאינדקס
for df_temp in [df_master_epi_demo, df_health_profile]:
    if 'location_key' not in df_temp.columns and df_temp.index.name == 'location_key':
        df_temp.reset_index(inplace=True)

year_col_epi = 'YEAR' if 'YEAR' in df_master_epi_demo.columns else 'year'

# =====================================================================
# 2. סינון שנים ומחשב את X4, X7 ואת משתנה המטרה Y
# =====================================================================
print("2. מסנן שנים ומחשב את X4, X7 ואת משתנה המטרה Y...")
df_2021_epi = df_master_epi_demo[df_master_epi_demo[year_col_epi] == 2021].copy()

# חישוב X7 (אחוז בני +60)
total_pop_60_plus = (
    df_2021_epi.get('population_age_60_69', 0) +
    df_2021_epi.get('population_age_70_79', 0) +
    df_2021_epi.get('population_age_80_and_older', 0)
)
df_2021_epi['X7_calculated'] = (total_pop_60_plus / df_2021_epi['population']) * 100

# חישוב ה-Y משנת 2022
df_2022_epi = df_master_epi_demo[df_master_epi_demo[year_col_epi] == 2022].copy()
df_2022_epi['excess_mortality_rate'] = (df_2022_epi['new_deceased'] / df_2022_epi['population']) * 100000
df_2022_clean = df_2022_epi[df_2022_epi['excess_mortality_rate'] > 0].copy()

global_median = df_2022_clean['excess_mortality_rate'].median()
df_2022_clean['Y_is_ready'] = (df_2022_clean['excess_mortality_rate'] <= global_median).astype(int)
df_Y = df_2022_clean[['location_key', 'Y_is_ready']].copy()

# בניית טבלת הבסיס האחודה
df_base_x = df_2021_epi[['location_key', 'X4', 'X7_calculated']].copy()

# התאמת סוג הנתונים לטקסט לצורך מיזוג בטוח
df_base_x['location_key'] = df_base_x['location_key'].astype(str).str.strip()
df_Y['location_key'] = df_Y['location_key'].astype(str).str.strip()

df_master_final = pd.merge(df_base_x, df_Y, on='location_key', how='inner')

# =====================================================================
# 3. מיזוג נתוני בריאות לטבלה האחודה (X1, X2)
# =====================================================================
print("3. מושך וממזג את נתוני המיטות (X1) והרופאים (X2)...")
beds_col = [col for col in df_health_profile.columns if 'bed' in col.lower()][0] if any('bed' in col.lower() for col in df_health_profile.columns) else None
physicians_col = [col for col in df_health_profile.columns if 'physician' in col.lower() or 'doctor' in col.lower()][0] if any('physician' in col.lower() or 'doctor' in col.lower() for col in df_health_profile.columns) else None

cols_to_pull_health = ['location_key']
rename_health_dict = {}

if beds_col:
    cols_to_pull_health.append(beds_col)
    rename_health_dict[beds_col] = 'X1_hospital_beds'
if physicians_col:
    cols_to_pull_health.append(physicians_col)
    rename_health_dict[physicians_col] = 'X2_physicians'

df_health_sub = df_health_profile[cols_to_pull_health].copy()
df_health_sub = df_health_sub.rename(columns=rename_health_dict)

# התאמת סוג הנתונים לטקסט לצורך מיזוג בטוח
df_health_sub['location_key'] = df_health_sub['location_key'].astype(str).str.strip()
df_master_final['location_key'] = df_master_final['location_key'].astype(str).str.strip()

df_master_final = pd.merge(df_master_final, df_health_sub, on='location_key', how='left')

# =====================================================================
# 4. מיזוג נתוני טכנולוגיה ו-GHS לטבלה האחודה (X3, X5, X6 - ללא X8 ו-X9)
# =====================================================================
print("4. מושך וממזג את נתוני הטכנולוגיה וה-GHS...")

year_col_tech = 'Year' if 'Year' in df_tech_ghs_raw.columns else 'YEAR' if 'YEAR' in df_tech_ghs_raw.columns else None
df_tech_2021 = df_tech_ghs_raw[df_tech_ghs_raw[year_col_tech] == 2021].copy() if year_col_tech else df_tech_ghs_raw.drop_duplicates(subset=['location_key'] if 'location_key' in df_tech_ghs_raw.columns else None).copy()

# איתור אוטומטי של עמודת המפתח בקובץ הטכנולוגיה
tech_key_col = None
potential_keys = ['location_key', 'location', 'country', 'key', 'code', 'index']

for p_key in potential_keys:
    found_cols = [col for col in df_tech_2021.columns if p_key in col.lower()]
    if found_cols:
        tech_key_col = found_cols[0]
        break

if not tech_key_col and df_tech_2021.index.name and any(p in df_tech_2021.index.name.lower() for p in potential_keys):
    df_tech_2021 = df_tech_2021.reset_index()
    tech_key_col = df_tech_2021.columns[0]

if not tech_key_col:
    df_tech_2021 = df_tech_2021.reset_index()
    tech_key_col = 'index'

cols_to_pull_tech = [tech_key_col]
rename_tech_dict = {tech_key_col: 'location_key'}

if 'OVERALL SCORE' in df_tech_2021.columns:
    cols_to_pull_tech.append('OVERALL SCORE')
    rename_tech_dict['OVERALL SCORE'] = 'X5_ghs_index'
if 'Internet users' in df_tech_2021.columns:
    cols_to_pull_tech.append('Internet users')
    rename_tech_dict['Internet users'] = 'X6_internet_users'

ghs_early_col = [col for col in df_tech_2021.columns if 'early' in col.lower() or 'detection' in col.lower()]
if ghs_early_col:
    cols_to_pull_tech.append(ghs_early_col[0])
    rename_tech_dict[ghs_early_col[0]] = 'X3_ghs_early_detection'

df_tech_sub = df_tech_2021[cols_to_pull_tech].copy()
df_tech_sub = df_tech_sub.rename(columns=rename_tech_dict)

# פתרון ה-ValueError: המרת מפתח המיזוג של קובץ הטכנולוגיה והטבלה המרכזית למחרוזות (טקסט)
df_tech_sub['location_key'] = df_tech_sub['location_key'].astype(str).str.strip()
df_master_final['location_key'] = df_master_final['location_key'].astype(str).str.strip()

df_master_final = pd.merge(df_master_final, df_tech_sub, on='location_key', how='left')

# =====================================================================
# 5. ארגון סופי, השלמת ערכים חסרים וייצוא הקובץ האחוד לדרייב
# =====================================================================
print("5. מנקה עמודות, מארגן את המטריצה האחודה וממלא חציונים...")

df_master_final = df_master_final.rename(columns={'X4': 'X4_testing_rate', 'X7_calculated': 'X7_age_60_plus'})

if 'X3_ghs_early_detection' not in df_master_final.columns: df_master_final['X3_ghs_early_detection'] = np.nan

# השלמת חציונים אוטומטית לעמודות מספריות (למניעת שגיאות במודלים)
numeric_cols = df_master_final.select_dtypes(include=['number']).columns
for col in numeric_cols:
    if col != 'Y_is_ready':
        df_master_final[col] = df_master_final[col].fillna(df_master_final[col].median())

# סדר העמודות הסופי בקובץ האחוד - כולל את X7 וללא X8 ו-X9
final_order = [
    'location_key',
    'X1_hospital_beds',
    'X2_physicians',
    'X3_ghs_early_detection',
    'X4_testing_rate',
    'X5_ghs_index',
    'X6_internet_users',
    'X7_age_60_plus',
    'Y_is_ready'
]
df_master_final = df_master_final[[col for col in final_order if col in df_master_final.columns]]

# ייצוא ושמירה של הקובץ האחוד כ-CSV לתוך הדרייב
output_file_path = os.path.join(data_dir, 'master_model_feature_matrix.csv')
df_master_final.to_csv(output_file_path, index=False)

print(f"\n--- 🌟 הקובץ האחוד המעודכן מוכן ומיוצא בהצלחה! 🌟 ---")
print(f"📁 נתיב שמירה בדרייב: {output_file_path}")
print(f"📊 מימדי הטבלה הסופיים (שורות, עמודות): {df_master_final.shape}")
display(df_master_final.head(15))


1. טוען את קבצי המקור המעודכנים...
קובץ הטכנולוגיה לא נמצא בשם ברירת המחדל, מציג קבצים קיימים ומנסה לאתר...
2. מסנן שנים ומחשב את X4, X7 ואת משתנה המטרה Y...
3. מושך וממזג את נתוני המיטות (X1) והרופאים (X2)...
4. מושך וממזג את נתוני הטכנולוגיה וה-GHS...
5. מנקה עמודות, מארגן את המטריצה האחודה וממלא חציונים...

--- 🌟 הקובץ האחוד המעודכן מוכן ומיוצא בהצלחה! 🌟 ---
📁 נתיב שמירה בדרייב: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/master_model_feature_matrix.csv
📊 מימדי הטבלה הסופיים (שורות, עמודות): (219, 8)


,location_key,X1_hospital_beds,X2_physicians,X3_ghs_early_detection,X4_testing_rate,X5_ghs_index,X7_age_60_plus,Y_is_ready
0,AD,2.3,3.33330,NaN,0.000000,NaN,36.201385,0
1,AE,2.3,2.52780,NaN,9210.469850,NaN,3.144635,1
2,AF,0.5,0.27820,NaN,0.000000,NaN,4.222826,1
3,AG,2.3,2.95600,NaN,0.000000,NaN,14.089944,0
4,AI,2.3,1.57655,NaN,0.000000,NaN,15.284629,0
5,AL,2.3,1.21640,NaN,411.206993,NaN,19.510960,1
6,AM,4.2,4.40230,NaN,667.976947,NaN,18.453723,0
7,AO,2.3,0.21460,NaN,0.000000,NaN,3.657909,1
8,AR,2.3,3.99010,NaN,380.272069,NaN,12.741438,0
9,AS,2.3,1.57655,NaN,0.000000,NaN,1036.665036,0


In [13]:
print(f"מספר המדינות הסופי במטריצה: {df_master_final.shape[0]}")


מספר המדינות הסופי במטריצה: 219
